# ML baselines (Week 4)

Predict **EOL** (cycles) from 44 early-cycle features in `cell_features.csv`.

| Step | Model | Status |
|------|-------|--------|
| 1 | Linear regression | below |
| 2 | ElasticNet (regularized linear) | below |
| 3 | Random forest | below |
| 4 | XGBoost | below |

If you know logistic regression: same workflow (features → fit → predict), but the target is a **number** (regression), not a class.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'data' / 'raw').is_dir():
            return candidate
    raise FileNotFoundError(f'Could not find data/raw/ starting from {here}')


ROOT = find_repo_root()
FEATURES_PATH = ROOT / 'data' / 'processed' / 'cell_features.csv'
SPLIT_PATH = ROOT / 'data' / 'processed' / 'cell_split.csv'
METRICS_PATH = ROOT / 'results' / 'metrics' / 'linear_regression.json'
FIGURE_PATH = ROOT / 'results' / 'figures' / 'pred_vs_true_eol_linear.png'
ENET_METRICS_PATH = ROOT / 'results' / 'metrics' / 'elasticnet.json'
ENET_FIGURE_PATH = ROOT / 'results' / 'figures' / 'pred_vs_true_eol_elasticnet.png'
RF_METRICS_PATH = ROOT / 'results' / 'metrics' / 'random_forest.json'
RF_FIGURE_PATH = ROOT / 'results' / 'figures' / 'pred_vs_true_eol_random_forest.png'
RF_IMPORTANCE_PATH = ROOT / 'results' / 'figures' / 'feature_importance_rf.png'
XGB_METRICS_PATH = ROOT / 'results' / 'metrics' / 'xgboost.json'
XGB_FIGURE_PATH = ROOT / 'results' / 'figures' / 'pred_vs_true_eol_xgboost.png'
XGB_IMPORTANCE_PATH = ROOT / 'results' / 'figures' / 'feature_importance_xgb.png'
COMPARE_FIGURE_PATH = ROOT / 'results' / 'figures' / 'model_comparison_baselines.png'
COMPARE_PRED_FIGURE_PATH = ROOT / 'results' / 'figures' / 'pred_vs_true_eol_baselines.png'

LABEL_COLS = ('file_id', 'cell_id', 'EOL', 'initial_capacity')
TARGET = 'EOL'
RANDOM_STATE = 42

print('Project root:', ROOT)
print('Features:', FEATURES_PATH)
print('Split:', SPLIT_PATH)

In [ ]:
cell_features = pd.read_csv(FEATURES_PATH)
feature_cols = [c for c in cell_features.columns if c not in LABEL_COLS]

assert len(cell_features) == 134
assert cell_features['file_id'].is_unique
assert cell_features[feature_cols].isna().sum().sum() == 0

print(f'Cells: {len(cell_features)}')
print(f'Features: {len(feature_cols)}')
print(f'EOL range: {cell_features[TARGET].min()} – {cell_features[TARGET].max()} cycles')

## Cell-level split (70 / 15 / 15)

Split by **`file_id`** so the same battery never appears in two sets.
Saved to `cell_split.csv` for Week 5+.

In [ ]:
if SPLIT_PATH.exists():
    split_df = pd.read_csv(SPLIT_PATH)
    assert set(split_df['split']) == {'train', 'val', 'test'}
    print('Loaded existing split from', SPLIT_PATH)
else:
    train_idx, holdout_idx = train_test_split(
        cell_features.index,
        test_size=40,
        random_state=RANDOM_STATE,
    )
    val_idx, test_idx = train_test_split(
        holdout_idx,
        test_size=20,
        random_state=RANDOM_STATE,
    )
    split_map = {}
    for idx in train_idx:
        split_map[idx] = 'train'
    for idx in val_idx:
        split_map[idx] = 'val'
    for idx in test_idx:
        split_map[idx] = 'test'

    split_df = cell_features[['file_id', 'cell_id']].copy()
    split_df['split'] = split_df.index.map(split_map)
    SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
    split_df.to_csv(SPLIT_PATH, index=False)
    print('Saved new split to', SPLIT_PATH)

counts = split_df['split'].value_counts().sort_index()
print(counts.to_string())
print(f'Total: {counts.sum()}')

In [ ]:
data = cell_features.merge(split_df[['file_id', 'split']], on='file_id', validate='one_to_one')

train = data[data['split'] == 'train']
val = data[data['split'] == 'val']
test = data[data['split'] == 'test']

X_train = train[feature_cols]
y_train = train[TARGET]
X_val = val[feature_cols]
y_val = val[TARGET]
X_test = test[feature_cols]
y_test = test[TARGET]

print(f'train {len(train)} | val {len(val)} | test {len(test)}')

## Metrics

- **MAE** — mean absolute error in cycles
- **RMSE** — root mean squared error (punishes large misses)
- **MAPE** — mean absolute percentage error vs true EOL

In [ ]:
def regression_metrics(y_true, y_pred) -> dict[str, float]:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    return {'mae': mae, 'rmse': rmse, 'mape': mape}


def evaluate(model, X, y) -> dict[str, float]:
    return regression_metrics(y, model.predict(X))

## Linear regression

`StandardScaler` + `LinearRegression` in a pipeline (scale features, then fit a straight-line model).
Comparable to logistic regression in spirit, but predicts EOL as a continuous cycle count.

In [ ]:
linear_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression()),
])

linear_model.fit(X_train, y_train)

metrics = {
    'model': 'linear_regression',
    'n_features': len(feature_cols),
    'split': {
        'train': len(train),
        'val': len(val),
        'test': len(test),
        'random_state': RANDOM_STATE,
    },
    'train': evaluate(linear_model, X_train, y_train),
    'val': evaluate(linear_model, X_val, y_val),
    'test': evaluate(linear_model, X_test, y_test),
}

pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'MAE': [metrics[s]['mae'] for s in ['train', 'val', 'test']],
    'RMSE': [metrics[s]['rmse'] for s in ['train', 'val', 'test']],
    'MAPE (%)': [metrics[s]['mape'] for s in ['train', 'val', 'test']],
}).round(2)

In [ ]:
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
METRICS_PATH.write_text(json.dumps(metrics, indent=2))
print('Saved metrics to', METRICS_PATH)

## Predicted vs true EOL (test set)

Points on the diagonal = perfect predictions.

In [ ]:
y_pred_test = linear_model.predict(X_test)
test_mae = metrics['test']['mae']

fig, ax = plt.subplots(figsize=(6, 6))
lo = min(y_test.min(), y_pred_test.min())
hi = max(y_test.max(), y_pred_test.max())
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='Perfect prediction')
ax.scatter(y_test, y_pred_test, alpha=0.85, edgecolors='white', linewidths=0.5)
ax.set_xlabel('True EOL (cycles)')
ax.set_ylabel('Predicted EOL (cycles)')
ax.set_title(f'Linear regression — test set (n={len(test)}, MAE={test_mae:.0f} cycles)')
ax.set_aspect('equal', adjustable='box')
ax.legend(loc='upper left')
fig.tight_layout()
FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_PATH, dpi=150)
plt.show()
print('Saved figure to', FIGURE_PATH)

## Step 2: ElasticNet

Like linear regression, but adds **L1 + L2 penalties** on feature weights — the regression analogue of regularized logistic regression. Useful when many features are correlated (e.g. our ΔV block).

- **`alpha`** — penalty strength (higher → simpler model, less overfitting)
- **`l1_ratio`** — mix of L1 vs L2 (`1.0` = Lasso-like sparsity, `0.0` = Ridge-like shrinkage)

Grid searches **alpha = 1 … 100** (very small alpha often fails to converge with 44 correlated features). Pick lowest **validation MAE**; report **test** once.

In [ ]:
# Grid from alpha=1 … 100 (weak penalties below 1 often fail to converge with 44 correlated features)
alphas = np.logspace(0, 2, 6)
l1_ratios = [0.1, 0.5, 0.9, 0.95, 1.0]
ENET_MAX_ITER = 100_000

best_val_mae = np.inf
best_params: dict[str, float] = {}
tuning_rows = []

for alpha in alphas:
    for l1_ratio in l1_ratios:
        candidate = Pipeline([
            ('scaler', StandardScaler()),
            ('model', ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=ENET_MAX_ITER,
                random_state=RANDOM_STATE,
            )),
        ])
        candidate.fit(X_train, y_train)
        val_scores = evaluate(candidate, X_val, y_val)
        tuning_rows.append({
            'alpha': alpha,
            'l1_ratio': l1_ratio,
            'val_mae': val_scores['mae'],
            'val_rmse': val_scores['rmse'],
        })
        if val_scores['mae'] < best_val_mae:
            best_val_mae = val_scores['mae']
            best_params = {'alpha': float(alpha), 'l1_ratio': l1_ratio}

print(f"Best on val: alpha={best_params['alpha']:.4g}, l1_ratio={best_params['l1_ratio']}")
pd.DataFrame(tuning_rows).sort_values('val_mae').head(10).round(2)

In [ ]:
elasticnet_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', ElasticNet(
        alpha=best_params['alpha'],
        l1_ratio=best_params['l1_ratio'],
        max_iter=ENET_MAX_ITER,
        random_state=RANDOM_STATE,
    )),
])
elasticnet_model.fit(X_train, y_train)

enet_metrics = {
    'model': 'elasticnet',
    'best_params': best_params,
    'n_features': len(feature_cols),
    'split': {
        'train': len(train),
        'val': len(val),
        'test': len(test),
        'random_state': RANDOM_STATE,
    },
    'train': evaluate(elasticnet_model, X_train, y_train),
    'val': evaluate(elasticnet_model, X_val, y_val),
    'test': evaluate(elasticnet_model, X_test, y_test),
}

pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'MAE': [enet_metrics[s]['mae'] for s in ['train', 'val', 'test']],
    'RMSE': [enet_metrics[s]['rmse'] for s in ['train', 'val', 'test']],
    'MAPE (%)': [enet_metrics[s]['mape'] for s in ['train', 'val', 'test']],
}).round(2)

In [ ]:
ENET_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
ENET_METRICS_PATH.write_text(json.dumps(enet_metrics, indent=2))
print('Saved metrics to', ENET_METRICS_PATH)

### Predicted vs true EOL — ElasticNet (test set)

In [ ]:
y_pred_enet = elasticnet_model.predict(X_test)
enet_test_mae = enet_metrics['test']['mae']

fig, ax = plt.subplots(figsize=(6, 6))
lo = min(y_test.min(), y_pred_enet.min())
hi = max(y_test.max(), y_pred_enet.max())
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='Perfect prediction')
ax.scatter(y_test, y_pred_enet, alpha=0.85, edgecolors='white', linewidths=0.5, color='C1')
ax.set_xlabel('True EOL (cycles)')
ax.set_ylabel('Predicted EOL (cycles)')
ax.set_title(f'ElasticNet — test set (n={len(test)}, MAE={enet_test_mae:.0f} cycles)')
ax.set_aspect('equal', adjustable='box')
ax.legend(loc='upper left')
fig.tight_layout()
ENET_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(ENET_FIGURE_PATH, dpi=150)
plt.show()
print('Saved figure to', ENET_FIGURE_PATH)

### Linear vs ElasticNet (test set)

Did regularization help on held-out cells?

In [ ]:
pd.DataFrame([
    {'model': 'linear_regression', **metrics['test']},
    {'model': 'elasticnet', **enet_metrics['test']},
]).set_index('model').round(2)

## Step 3: Random forest

Many decision trees averaged together — can capture **nonlinear** patterns. No feature scaling needed. Tune `max_depth` and `min_samples_leaf` on **val**; report **test** once.

In [ ]:
max_depths = [5, 10, 15, None]
min_samples_leaves = [1, 3, 5]

best_val_mae = np.inf
best_rf_params: dict = {}
rf_tuning_rows = []

for max_depth in max_depths:
    for min_samples_leaf in min_samples_leaves:
        candidate = RandomForestRegressor(
            n_estimators=300,
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        candidate.fit(X_train, y_train)
        val_scores = evaluate(candidate, X_val, y_val)
        rf_tuning_rows.append({
            'max_depth': max_depth,
            'min_samples_leaf': min_samples_leaf,
            'val_mae': val_scores['mae'],
        })
        if val_scores['mae'] < best_val_mae:
            best_val_mae = val_scores['mae']
            best_rf_params = {
                'max_depth': max_depth,
                'min_samples_leaf': min_samples_leaf,
            }

print(f"Best on val: {best_rf_params}")
pd.DataFrame(rf_tuning_rows).sort_values('val_mae').head(10).round(2)

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=best_rf_params['max_depth'],
    min_samples_leaf=best_rf_params['min_samples_leaf'],
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

rf_metrics = {
    'model': 'random_forest',
    'best_params': best_rf_params,
    'n_features': len(feature_cols),
    'split': {
        'train': len(train),
        'val': len(val),
        'test': len(test),
        'random_state': RANDOM_STATE,
    },
    'train': evaluate(rf_model, X_train, y_train),
    'val': evaluate(rf_model, X_val, y_val),
    'test': evaluate(rf_model, X_test, y_test),
}

pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'MAE': [rf_metrics[s]['mae'] for s in ['train', 'val', 'test']],
    'RMSE': [rf_metrics[s]['rmse'] for s in ['train', 'val', 'test']],
    'MAPE (%)': [rf_metrics[s]['mape'] for s in ['train', 'val', 'test']],
}).round(2)

In [ ]:
RF_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
RF_METRICS_PATH.write_text(json.dumps(rf_metrics, indent=2))
print('Saved metrics to', RF_METRICS_PATH)

In [ ]:
y_pred_rf = rf_model.predict(X_test)
rf_test_mae = rf_metrics['test']['mae']

fig, ax = plt.subplots(figsize=(6, 6))
lo = min(y_test.min(), y_pred_rf.min())
hi = max(y_test.max(), y_pred_rf.max())
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='Perfect prediction')
ax.scatter(y_test, y_pred_rf, alpha=0.85, edgecolors='white', linewidths=0.5, color='C2')
ax.set_xlabel('True EOL (cycles)')
ax.set_ylabel('Predicted EOL (cycles)')
ax.set_title(f'Random forest — test set (n={len(test)}, MAE={rf_test_mae:.0f} cycles)')
ax.set_aspect('equal', adjustable='box')
ax.legend(loc='upper left')
fig.tight_layout()
RF_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(RF_FIGURE_PATH, dpi=150)
plt.show()
print('Saved figure to', RF_FIGURE_PATH)

In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot.barh(ax=ax, color='C2')
ax.set_xlabel('Feature importance (Gini)')
ax.set_title('Random forest — top 15 features')
fig.tight_layout()
RF_IMPORTANCE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(RF_IMPORTANCE_PATH, dpi=150)
plt.show()
print('Saved figure to', RF_IMPORTANCE_PATH)

In [ ]:
pd.DataFrame([
    {'model': 'linear_regression', **metrics['test']},
    {'model': 'elasticnet', **enet_metrics['test']},
    {'model': 'random_forest', **rf_metrics['test']},
]).set_index('model').round(2)

## Step 4: XGBoost

Gradient-boosted trees — often strong on tabular data. Tune `max_depth` and `learning_rate` on **val**; report **test** once.

In [ ]:
max_depths = [3, 5, 7]
learning_rates = [0.05, 0.1]

best_val_mae = np.inf
best_xgb_params: dict = {}
xgb_tuning_rows = []

for max_depth in max_depths:
    for learning_rate in learning_rates:
        candidate = XGBRegressor(
            objective='reg:squarederror',
            n_estimators=300,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
        candidate.fit(X_train, y_train)
        val_scores = evaluate(candidate, X_val, y_val)
        xgb_tuning_rows.append({
            'max_depth': max_depth,
            'learning_rate': learning_rate,
            'val_mae': val_scores['mae'],
        })
        if val_scores['mae'] < best_val_mae:
            best_val_mae = val_scores['mae']
            best_xgb_params = {
                'max_depth': max_depth,
                'learning_rate': learning_rate,
            }

print(f"Best on val: {best_xgb_params}")
pd.DataFrame(xgb_tuning_rows).sort_values('val_mae').head(10).round(2)

In [ ]:
xgb_model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=300,
    max_depth=best_xgb_params['max_depth'],
    learning_rate=best_xgb_params['learning_rate'],
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
xgb_model.fit(X_train, y_train)

xgb_metrics = {
    'model': 'xgboost',
    'best_params': best_xgb_params,
    'n_features': len(feature_cols),
    'split': {
        'train': len(train),
        'val': len(val),
        'test': len(test),
        'random_state': RANDOM_STATE,
    },
    'train': evaluate(xgb_model, X_train, y_train),
    'val': evaluate(xgb_model, X_val, y_val),
    'test': evaluate(xgb_model, X_test, y_test),
}

pd.DataFrame({
    'split': ['train', 'val', 'test'],
    'MAE': [xgb_metrics[s]['mae'] for s in ['train', 'val', 'test']],
    'RMSE': [xgb_metrics[s]['rmse'] for s in ['train', 'val', 'test']],
    'MAPE (%)': [xgb_metrics[s]['mape'] for s in ['train', 'val', 'test']],
}).round(2)

In [ ]:
XGB_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
XGB_METRICS_PATH.write_text(json.dumps(xgb_metrics, indent=2))
print('Saved metrics to', XGB_METRICS_PATH)

In [ ]:
y_pred_xgb = xgb_model.predict(X_test)
xgb_test_mae = xgb_metrics['test']['mae']

fig, ax = plt.subplots(figsize=(6, 6))
lo = min(y_test.min(), y_pred_xgb.min())
hi = max(y_test.max(), y_pred_xgb.max())
ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, label='Perfect prediction')
ax.scatter(y_test, y_pred_xgb, alpha=0.85, edgecolors='white', linewidths=0.5, color='C3')
ax.set_xlabel('True EOL (cycles)')
ax.set_ylabel('Predicted EOL (cycles)')
ax.set_title(f'XGBoost — test set (n={len(test)}, MAE={xgb_test_mae:.0f} cycles)')
ax.set_aspect('equal', adjustable='box')
ax.legend(loc='upper left')
fig.tight_layout()
XGB_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(XGB_FIGURE_PATH, dpi=150)
plt.show()
print('Saved figure to', XGB_FIGURE_PATH)

In [ ]:
xgb_importance = pd.Series(
    xgb_model.feature_importances_, index=feature_cols
).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(8, 5))
xgb_importance.plot.barh(ax=ax, color='C3')
ax.set_xlabel('Feature importance (gain)')
ax.set_title('XGBoost — top 15 features')
fig.tight_layout()
XGB_IMPORTANCE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(XGB_IMPORTANCE_PATH, dpi=150)
plt.show()
print('Saved figure to', XGB_IMPORTANCE_PATH)

## All baselines — test set comparison

In [ ]:
comparison = pd.DataFrame([
    {'model': 'linear_regression', **metrics['test']},
    {'model': 'elasticnet', **enet_metrics['test']},
    {'model': 'random_forest', **rf_metrics['test']},
    {'model': 'xgboost', **xgb_metrics['test']},
]).set_index('model').round(2)

comparison

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
comparison['mae'].sort_values().plot.bar(ax=ax, color=['C0', 'C1', 'C2', 'C3'])
ax.set_ylabel('Test MAE (cycles)')
ax.set_title('Model comparison — holdout test set')
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha='right')
fig.tight_layout()
COMPARE_FIGURE_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(COMPARE_FIGURE_PATH, dpi=150)
plt.show()
print('Saved figure to', COMPARE_FIGURE_PATH)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 10), sharex=True, sharey=True)
preds = [
    ('Linear', linear_model.predict(X_test), metrics['test']['mae'], 'C0'),
    ('ElasticNet', elasticnet_model.predict(X_test), enet_metrics['test']['mae'], 'C1'),
    ('Random forest', rf_model.predict(X_test), rf_metrics['test']['mae'], 'C2'),
    ('XGBoost', xgb_model.predict(X_test), xgb_metrics['test']['mae'], 'C3'),
]
lo, hi = y_test.min(), y_test.max()

for ax, (name, y_pred, mae, color) in zip(axes.ravel(), preds):
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1)
    ax.scatter(y_test, y_pred, alpha=0.85, edgecolors='white', linewidths=0.5, color=color)
    ax.set_title(f'{name} (MAE={mae:.0f})')
    ax.set_aspect('equal', adjustable='box')

axes[1, 0].set_xlabel('True EOL (cycles)')
axes[1, 1].set_xlabel('True EOL (cycles)')
axes[0, 0].set_ylabel('Predicted EOL (cycles)')
axes[1, 0].set_ylabel('Predicted EOL (cycles)')
fig.suptitle('Predicted vs true EOL — all baselines (test set)', y=1.02)
fig.tight_layout()
fig.savefig(COMPARE_PRED_FIGURE_PATH, dpi=150, bbox_inches='tight')
plt.show()
print('Saved figure to', COMPARE_PRED_FIGURE_PATH)